# Pretrain experiment

Runs enhanced (unified masked) pretraining and writes everything to
`experiments/<experiment_name>/`:

- `pretrain_config.yaml` — frozen copy of the merged config (base YAML + `overrides`)
- `pretrain_overrides.yaml` — just the dict you passed below, for at-a-glance diffing
- `pretrain_metadata.json` — timestamp, git SHA, resolved device, final/best loss, wallclock, the overrides inline
- `pretrain.log` — full training log (includes GPU diagnostics at startup)
- `checkpoints/checkpoint_epoch_*.pth` — saved encoder weights
- `README.md` — name + description (the short note you write below)

Edit the **Parameters** cell, run all cells. Re-running with the same
`experiment_name` will raise unless you pass `overwrite=True`.

The `experiment_name` is the only key you need to pass to the eval notebook —
it auto-loads the architecture from this experiment's saved config.

In [1]:
# === Parameters ===
experiment_name = "houston_enhanced_spatial_mask_test_run1_seed52"
description = (
    "Houston pretraining with the re-tuned config (Houston-proven recipe: "
    "2 heads / 2 layers, lambda 0.5, band-only masking, AMP off, deterministic)."
)

# Path to the YAML config to start from.
config_path = "configs/pretrain/houston_pretrain_enhanced.yaml"

# Which device/GPU to train on. Mirrors `hardware.device` in the config and
# overrides whatever the YAML sets. Options:
#   "cuda"    -> first visible GPU (cuda:0); errors if no CUDA.
#   "cuda:N"  -> specific GPU index (e.g. "cuda:1", "cuda:2", "cuda:3").
#   "auto"    -> cuda:0 if available, else cpu.
#   "cpu"     -> force CPU.
# Run `!nvidia-smi` in a cell to see which GPUs are free.
device = "cuda:3"

# Dict-shaped overrides applied (deep-merge) on top of the loaded YAML.
# Multiple sections can be combined in one dict. Anything you set here is
# also recorded verbatim to experiments/<name>/pretrain_overrides.yaml so
# it's obvious what changed from the base config.
#
# NOTE: architecture, masking regime, epochs, AMP and determinism now all live
# in houston_pretrain_enhanced.yaml (band-only masking: band_mask_ratio=0.9,
# spatial_mask_ratio=0.0). Leave the pretrain overrides empty so the config
# drives the run. Do NOT re-introduce a spatial_mask_ratio override here — the
# earlier Houston runs forced spatial_mask_ratio=0.75, diverging from the recipe.
#
# Uncomment any of the blocks below only if you deliberately want to experiment.
overrides = {
    # --- Training schedule ---
     "pretrain": {"epochs": 2000, "warmup_epochs": 100, "batch_size": 128,"band_mask_ratio": 0.0, "spatial_mask_ratio": 0.75},

    # --- Masking ratios (config default is band-only: 0.9 / 0.0) ---
    # "pretrain": {"band_mask_ratio": 0.9, "spatial_mask_ratio": 0.0},

    # --- Loss weighting (center-weighted reconstruction) ---
    # "pretrain": {"recon_center_sigma": 1.0},

    # --- Optimizer ---
    # "pretrain": {"lr": 3e-4, "weight_decay": 0.1, "adam_betas": [0.9, 0.999],
    #              "warmup_start_factor": 0.001, "grad_clip": 0.5},

    # --- Model capacity (must match in eval; eval auto-reads these) ---
    # "model": {"embed_dim": 192, "num_heads": 4, "num_layers": 6,
    #           "proj_hidden_dim": 768, "proj_l2_normalize": False},

    # --- Hardware (note: `device` above already handles GPU selection) ---
    "hardware": {"seed": 42},
    # "data":     {"num_workers": 8, "persistent_workers": True, "prefetch_factor": 4},
}

# Resume from a checkpoint inside this experiment (or another).
# Pass an absolute or repo-relative path, or None to start fresh.
resume = None

overwrite = False  # Whether to overwrite existing experiment with the same name.

In [2]:
import os, sys
from pathlib import Path

# Make repo root importable regardless of where Jupyter was launched.
REPO = Path.cwd()
while not (REPO / "lib" / "experiments.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("Could not locate repo root containing lib/experiments.py")
    REPO = REPO.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Repo root:", REPO)

Repo root: /work/nmaric/CoFFE/CoFFE


In [3]:
from lib.pretrain_runner import run_pretrain

# Inject the `device` parameter into overrides so it lands in
# pretrain_overrides.yaml alongside the rest of the run's settings.
run_overrides = {**overrides, "hardware": {**overrides.get("hardware", {}), "device": device}}

exp = run_pretrain(
    name=experiment_name,
    description=description,
    config=config_path,
    overrides=run_overrides,
    resume=resume,
    overwrite=overwrite,
)
print("Experiment dir:", exp.root)

Experiment dir: /work/nmaric/CoFFE/CoFFE/experiments/houston_enhanced_spatial_run3


In [ ]:
# Verify what actually got recorded for this run: the resolved device
# (cuda:0 / cpu) and the overrides exactly as they were saved to disk.
import json
print("Resolved device:", exp.metadata.get("resolved_device"),
      "|", exp.metadata.get("cuda_device_name", ""))
print("Overrides applied:")
print(json.dumps(exp.metadata.get("overrides", {}), indent=2))
print("\nFull metadata:")
print(json.dumps(exp.metadata, indent=2, default=str))

Resolved device: cuda:3 | NVIDIA GeForce RTX 4090
Overrides applied:
{
  "pretrain": {
    "epochs": 2000,
    "warmup_epochs": 100,
    "batch_size": 128,
    "band_mask_ratio": 0.0,
    "spatial_mask_ratio": 0.75
  },
  "hardware": {
    "seed": 42,
    "device": "cuda:3"
  }
}

Full metadata:
{
  "name": "houston_enhanced_spatial_run3",
  "description": "Houston pretraining with the re-tuned config (Houston-proven recipe: 2 heads / 2 layers, lambda 0.5, band-only masking, AMP off, deterministic).",
  "status": "complete",
  "started_at": "2026-06-09T15:47:14",
  "git_sha": "08b592987d56d1d94aace857ba6a0d200215c7ae",
  "base_config_path": "configs/pretrain/houston_pretrain_enhanced.yaml",
  "overrides": {
    "pretrain": {
      "epochs": 2000,
      "warmup_epochs": 100,
      "batch_size": 128,
      "band_mask_ratio": 0.0,
      "spatial_mask_ratio": 0.75
    },
    "hardware": {
      "seed": 42,
      "device": "cuda:3"
    }
  },
  "finished_at": "2026-06-09T16:37:05",
  "wallc

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


In [8]:
# List of checkpoints written by this run.
for p in sorted(exp.checkpoints_dir.glob("*.pth")):
    print(p.name, f"{p.stat().st_size/1e6:.1f} MB")

checkpoint_epoch_1000.pth 8.3 MB
checkpoint_epoch_1200.pth 8.3 MB
checkpoint_epoch_1400.pth 8.3 MB
checkpoint_epoch_1600.pth 8.3 MB
checkpoint_epoch_1800.pth 8.3 MB
checkpoint_epoch_200.pth 8.3 MB
checkpoint_epoch_2000.pth 8.3 MB
checkpoint_epoch_400.pth 8.3 MB
checkpoint_epoch_600.pth 8.3 MB
checkpoint_epoch_800.pth 8.3 MB
encoder_final.pth 2.4 MB
final.pth 8.3 MB


In [9]:
import gc, torch

# Drop any Python references to the model/runner first
del exp
try:
    del run_overrides
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()  # optional, releases shared memory handles

# Confirm it actually freed
print(torch.cuda.memory_summary(device=device, abbreviated=True))


|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 3                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  63744 KiB | 613247 KiB | 515173 GiB | 515173 GiB |
|---------------------------------------------------------------------------|
| Active memory         |  63744 KiB | 613247 KiB | 515173 GiB | 515173 GiB |
|---------------------------------------------------------------------------|
| Requested memory      |  63424 KiB | 605515 KiB | 500350 GiB | 500350 GiB |
|---------------------------------------------------------------